# Model 03: time-varying connectivity

This notebook lets a migration barrier appear or disappear over time. It tracks genealogical ancestry only, not DNA.

In [ ]:
import os, sys, subprocess
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/Evolution-Creation'):
        subprocess.run(['git','clone','-q','https://github.com/vafaei-ar/Evolution-Creation.git','/content/Evolution-Creation'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-e','/content/Evolution-Creation'], check=True)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from evolution_creation.structured import make_linear_migration_matrix
from evolution_creation.temporal import make_barrier_schedule, make_constant_schedule, simulate_time_varying_replicates


In [ ]:
n_demes=widgets.IntSlider(value=6,min=2,max=8,description='Demes')
deme_size=widgets.IntSlider(value=150,min=50,max=600,step=25,description='Size/deme')
migration=widgets.FloatSlider(value=0.02,min=0,max=0.15,step=0.001,readout_format='.3f',description='Migration')
generations=widgets.IntSlider(value=80,min=10,max=150,step=5,description='Generations')
founders=widgets.IntSlider(value=12,min=1,max=50,description='Founders')
barrier_edge=widgets.IntSlider(value=3,min=1,max=7,description='After deme')
mode=widgets.Dropdown(options=[('Always connected','connected'),('Always isolated','closed'),('Barrier closes later','closes'),('Barrier opens later','opens'),('Temporary barrier','temporary')],value='closes',description='Scenario')
switch1=widgets.IntSlider(value=20,min=1,max=150,description='Switch 1')
switch2=widgets.IntSlider(value=45,min=2,max=151,description='Switch 2')
replicates=widgets.IntSlider(value=100,min=10,max=300,step=10,description='Replicates')
seed=widgets.IntText(value=20260920,description='Seed')
display(n_demes,deme_size,migration,generations,founders,barrier_edge,mode,switch1,switch2,replicates,seed)

In [ ]:
def build_schedule():
    base=make_linear_migration_matrix(n_demes.value,migration.value)
    edge=min(barrier_edge.value-1,n_demes.value-2)
    g=generations.value
    if mode.value=='connected': return make_constant_schedule(base,g),[]
    if mode.value=='closed': return make_barrier_schedule(base,g,edge,1),[1]
    if mode.value=='closes':
        s=min(switch1.value,g+1); return make_barrier_schedule(base,g,edge,s),[s]
    if mode.value=='opens':
        e=min(switch1.value,g+1); return make_barrier_schedule(base,g,edge,1,e),[e]
    s=min(switch1.value,g+1); e=min(switch2.value,g+1)
    if e<=s: raise ValueError('Switch 2 must be later than Switch 1')
    return make_barrier_schedule(base,g,edge,s,e),[s,e]

def run_model(_=None):
    schedule,switches=build_schedule()
    curves=simulate_time_varying_replicates([deme_size.value]*n_demes.value,schedule,founder_count=min(founders.value,deme_size.value),replicates=replicates.value,seed=seed.value)
    median=np.median(curves,axis=0); x=np.arange(generations.value+1)
    fig,ax=plt.subplots(figsize=(10,5))
    for d in range(n_demes.value): ax.plot(x,median[:,d],label=f'Deme {d+1}')
    for s in switches:
        if s<=generations.value: ax.axvline(s,linestyle='--',alpha=.7)
    ax.set(xlabel='Generation',ylabel='Median fraction with founder ancestry',ylim=(0,1.02)); ax.legend(ncol=2); plt.show()
    final=curves[:,-1,:]
    print(f'Every deme reached: {np.all(final>0,axis=1).mean():.1%}')
    print(f'Global genealogical fixation: {np.all(final==1,axis=1).mean():.1%}')

button=widgets.Button(description='Run simulation',button_style='primary'); button.on_click(run_model); display(button); run_model()

## Interpretation

A barrier that appears after ancestry has crossed cannot erase that prior genealogical connection. A barrier present from the start can make spread impossible. Timing is part of the hypothesis.